# Tune CatBoost on the streaming features

Reuses `stream_feats.npz` from `gbt_blend.ipynb` (deployed 43-feature StreamState). Baselines: LGB 0.6007, Cat 0.6066, LGB+Cat 0.6076. Search Cat hyperparams, report best Cat and best LGB+Cat blend.

Caveat: tuning on the va holdout is mildly optimistic — grid kept small to limit holdout-overfit. Final pick still needs a clean test.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys, time, json
import numpy as np, pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier

HERE = os.path.abspath("")
EXPERIMENTS = os.path.abspath(os.path.join(HERE, "..", ".."))
sys.path.insert(0, EXPERIMENTS)
import sb_common as C
CACHE = os.path.join(HERE, "stream_feats.npz")

Z = np.load(CACHE, allow_pickle=True)
Xtr, ytr, sids = Z["Xtr"], Z["ytr"], Z["sids"]
Xva, va_id, va_time = Z["Xva"], Z["va_id"], Z["va_time"]
va_index = pd.MultiIndex.from_arrays([va_id, va_time], names=["id", "time"])
yva = pd.Series(C.load_y().reindex(va_index).to_numpy().astype(np.int8), index=va_index)
cmap = dict(zip(*np.unique(sids, return_counts=True)))
w = np.array([1.0 / cmap[s] for s in sids]); w /= w.mean()
def auc(p): return C.ts_auc(pd.Series(p, index=va_index), yva)

# fixed LGB (deployed config) for blend eval
lgb_m = lgb.LGBMClassifier(objective="binary", n_estimators=1500, learning_rate=0.02,
                           num_leaves=63, min_child_samples=300, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbosity=-1,
                           random_state=0).fit(Xtr, ytr, sample_weight=w)
p_lgb = lgb_m.predict_proba(Xva)[:, 1]; r_lgb = C.rank01(p_lgb)
print(f"LGB {auc(p_lgb):.4f}  | tr {Xtr.shape} va {Xva.shape}")

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGB 0.6007  | tr (263219, 43) va (1012236, 43)


In [2]:
def fit_cat(**kw):
    base = dict(loss_function="Logloss", random_seed=0, verbose=0, thread_count=-1,
                bootstrap_type="Bernoulli", subsample=0.8)
    base.update(kw)
    m = CatBoostClassifier(**base).fit(Xtr, ytr, sample_weight=w)
    return m.predict_proba(Xva)[:, 1]

configs = {
    "base(d6,l3,it1500,lr.02)": dict(depth=6, l2_leaf_reg=3, iterations=1500, learning_rate=0.02),
    "d7":                       dict(depth=7, l2_leaf_reg=3, iterations=1500, learning_rate=0.02),
    "d8":                       dict(depth=8, l2_leaf_reg=3, iterations=1500, learning_rate=0.02),
    "d7_l6":                    dict(depth=7, l2_leaf_reg=6, iterations=1500, learning_rate=0.02),
    "d7_l10":                   dict(depth=7, l2_leaf_reg=10, iterations=1500, learning_rate=0.02),
    "d7_it3000_lr.01":          dict(depth=7, l2_leaf_reg=6, iterations=3000, learning_rate=0.01),
    "d7_it2500_lr.02":          dict(depth=7, l2_leaf_reg=6, iterations=2500, learning_rate=0.02),
    "d7_border254":             dict(depth=7, l2_leaf_reg=6, iterations=1500, learning_rate=0.02, border_count=254),
    "d7_rs2":                   dict(depth=7, l2_leaf_reg=6, iterations=1500, learning_rate=0.02, random_strength=2.0),
}

results = {}
t0 = time.time()
for name, cfg in configs.items():
    p = fit_cat(**cfg)
    a_cat = auc(p); a_bl = auc(r_lgb + C.rank01(p))
    results[name] = {"cat": a_cat, "blend": a_bl, "cfg": cfg}
    print(f"  {name:26s} Cat {a_cat:.4f}  LGB+Cat {a_bl:.4f}   {time.time()-t0:.0f}s", flush=True)

  base(d6,l3,it1500,lr.02)   Cat 0.6066  LGB+Cat 0.6076   37s


  d7                         Cat 0.6069  LGB+Cat 0.6072   79s


  d8                         Cat 0.6003  LGB+Cat 0.6040   129s


  d7_l6                      Cat 0.6075  LGB+Cat 0.6076   230s


  d7_l10                     Cat 0.6076  LGB+Cat 0.6076   293s


  d7_it3000_lr.01            Cat 0.6061  LGB+Cat 0.6068   405s


  d7_it2500_lr.02            Cat 0.6071  LGB+Cat 0.6070   501s


  d7_border254               Cat 0.6075  LGB+Cat 0.6076   558s


  d7_rs2                     Cat 0.6066  LGB+Cat 0.6068   615s


In [3]:
best_cat = max(results, key=lambda k: results[k]["cat"])
best_bl = max(results, key=lambda k: results[k]["blend"])
print("baselines: LGB 0.6007 | Cat(base) %.4f | LGB+Cat(base) %.4f" % (
    results["base(d6,l3,it1500,lr.02)"]["cat"], results["base(d6,l3,it1500,lr.02)"]["blend"]))
print(f"BEST Cat single: {best_cat}  {results[best_cat]['cat']:.4f}")
print(f"BEST LGB+Cat   : {best_bl}  {results[best_bl]['blend']:.4f}")
json.dump({k: {"cat": v["cat"], "blend": v["blend"], "cfg": v["cfg"]} for k, v in results.items()},
          open(os.path.join(HERE, "catboost_tune_result.json"), "w"), indent=2)
print("saved")

baselines: LGB 0.6007 | Cat(base) 0.6066 | LGB+Cat(base) 0.6076
BEST Cat single: d7_l10  0.6076
BEST LGB+Cat   : d7_l6  0.6076
saved
